# 💻 Chapter 21: Statistical Testing for Feature Flags
*Track 2 — Developers | Tier 2*

> **🎬 Hook:** You're running a feature flag at 10%. You see metrics improve. At what rollout % do you call it a win and ship to 100%?

**🎯 Objectives:** Apply sequential testing to feature flags; implement multi-armed bandits; avoid the peeking problem properly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# ── Sequential Probability Ratio Test (SPRT) ──
# Allows peeking with controlled error rates

def sprt_test(data_a, data_b, p0_effect=0, p1_effect=0.01, alpha=0.05, beta=0.20):
    # Wald's SPRT: stop when likelihood ratio crosses a threshold.
    A = np.log((1-beta)/alpha)      # upper boundary (reject H0)
    B = np.log(beta/(1-alpha))       # lower boundary (accept H0)

    log_lr = 0
    decisions = []
    for i in range(max(len(data_a), len(data_b))):
        if i < len(data_a) and i < len(data_b):
            xa = data_a[i]; xb = data_b[i]
            # Update log likelihood ratio
            if xa != xb:
                if xb == 1:
                    log_lr += np.log((p1_effect + 0.5) / (p0_effect + 0.5 + 1e-10))
                else:
                    log_lr += np.log((1-p1_effect-0.5+1e-10) / (1-p0_effect-0.5+1e-10))
        if log_lr >= A:
            decisions.append('reject'); break
        elif log_lr <= B:
            decisions.append('accept'); break
        decisions.append('continue')
    return decisions, log_lr

# Simulate feature flag test
n = 2000
true_rate_control  = 0.050
true_rate_treatment = 0.058  # real improvement

control   = rng.binomial(1, true_rate_control, n)
treatment = rng.binomial(1, true_rate_treatment, n)

# Fixed horizon test (bad — peeking)
decisions_fixed = []
for n_obs in range(10, n, 10):
    _, p = stats.ttest_ind(treatment[:n_obs], control[:n_obs])
    decisions_fixed.append(p < 0.05)

# SPRT (proper sequential test)
decisions_seq, _ = sprt_test(control, treatment)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(10, n, 10), [int(d) for d in decisions_fixed],
        'r-', lw=2, label='Fixed-horizon peeking (WRONG)', alpha=0.7)
ax.set_xlabel('Observations so far'); ax.set_ylabel('Significant (1=Yes)')
ax.set_title('Feature Flag: Peeking vs Sequential Testing', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('ch21_feature_flags.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Fixed-horizon: 'significant' at {sum(decisions_fixed)} of {len(decisions_fixed)} peeks")
print(f"  This inflates Type I error!")

In [ ]:
# ── Multi-Armed Bandit for Feature Flag Rollout ──
class EpsilonGreedy:
    def __init__(self, n_arms, epsilon=0.1):
        self.n_arms = n_arms; self.epsilon = epsilon
        self.counts = np.zeros(n_arms); self.rewards = np.zeros(n_arms)

    def select_arm(self):
        if rng.random() < self.epsilon:
            return rng.integers(self.n_arms)  # explore
        return np.argmax(self.rewards / np.maximum(self.counts, 1))  # exploit

    def update(self, arm, reward):
        self.counts[arm] += 1
        self.rewards[arm] += reward

class ThompsonSampling:
    def __init__(self, n_arms):
        self.n_arms = n_arms
        self.alphas = np.ones(n_arms); self.betas = np.ones(n_arms)

    def select_arm(self):
        samples = [rng.beta(self.alphas[i], self.betas[i]) for i in range(self.n_arms)]
        return np.argmax(samples)

    def update(self, arm, reward):
        self.alphas[arm] += reward; self.betas[arm] += 1-reward

# True conversion rates for 3 variants
true_rates = [0.050, 0.065, 0.058]  # B is best
n_rounds = 5000

for BanditClass, name in [(EpsilonGreedy, 'ε-Greedy'), (ThompsonSampling, 'Thompson Sampling')]:
    if name == 'ε-Greedy':
        bandit = EpsilonGreedy(3, epsilon=0.1)
    else:
        bandit = ThompsonSampling(3)

    arm_counts = np.zeros(3)
    for _ in range(n_rounds):
        arm = bandit.select_arm()
        reward = int(rng.random() < true_rates[arm])
        bandit.update(arm, reward)
        arm_counts[arm] += 1

    print(f"
{name}:")
    for i, count in enumerate(arm_counts):
        rate = count/n_rounds
        print(f"  Variant {i+1} (true={true_rates[i]:.3f}): {count:.0f} users ({rate:.1%})")
    optimal_pct = arm_counts[np.argmax(true_rates)] / n_rounds
    print(f"  → Sent {optimal_pct:.1%} to best variant!")

## ✏️ Section 6 — Coding Challenges

**Challenge 1:** Implement the Bonferroni correction for testing 5 metrics simultaneously.
Adjust α and recalculate required sample sizes.

**Challenge 2:** Build a simple feature flag system that uses Thompson Sampling to gradually shift traffic to the winning variant.

<details><summary>Solutions</summary>See patterns in the code above.</details>

In [ ]:
# Bonferroni correction
n_tests = 5
alpha_family = 0.05
alpha_individual = alpha_family / n_tests
print(f"Testing {n_tests} metrics:")
print(f"  Family-wise α = {alpha_family}")
print(f"  Bonferroni-corrected α per test = {alpha_individual}")
print(f"  FWER without correction: {1-(1-alpha_family)**n_tests:.3f}")

# Sample size with corrected alpha
from scipy.stats import norm
def sample_size(p1, p2, alpha, power):
    z_alpha = norm.ppf(1-alpha/2)
    z_beta = norm.ppf(power)
    p_bar = (p1 + p2) / 2
    return int(np.ceil(2 * (z_alpha+z_beta)**2 * p_bar*(1-p_bar) / (p1-p2)**2))

n_uncorrected = sample_size(0.05, 0.06, 0.05, 0.80)
n_corrected   = sample_size(0.05, 0.06, alpha_individual, 0.80)
print(f"
Sample size per variant:")
print(f"  Without Bonferroni: {n_uncorrected:,}")
print(f"  With Bonferroni:    {n_corrected:,} ({n_corrected/n_uncorrected:.1f}x larger)")

## 🎯 Recap
1. Standard peeking inflates Type I error — use SPRT for sequential testing.
2. Multi-armed bandits (ε-greedy, Thompson Sampling) balance exploration/exploitation.
3. Multiple metrics → Bonferroni correction or FDR control.

**Next:** [Chapter 22 — Debugging Probabilistic Systems]